In [1]:
import pandas as pd
import numpy as np

In [2]:
df_gain = pd.read_csv(r"D:\UST Project\UST_Analog_automation\data\processed\UpdatedOpam\gain_ml.csv")
df_pmf  = pd.read_csv(r"D:\UST Project\UST_Analog_automation\data\processed\UpdatedOpam\pm_ml.csv")
df_ugf  = pd.read_csv(r"D:\UST Project\UST_Analog_automation\data\processed\UpdatedOpam\ugf_ml.csv")

In [3]:
violations = (
    df_pmf.groupby(["a", "b", "c", "d"])["pm"]
      .nunique()
      .reset_index()
      .query("pm > 1")
)

print("Number of violating input combinations:", len(violations))


Number of violating input combinations: 0


In [4]:
df_clean = df_gain.drop_duplicates(subset=["a", "b", "c", "d"], keep="first")


In [5]:
bad_rows = df_pmf.merge(
    violations[["a", "b", "c", "d"]],
    on=["a", "b", "c", "d"],
    how="inner"
)

bad_rows.sort_values(["a", "b", "c", "d"])


,a,b,c,d,pm


In [6]:
print(df_gain.shape)

(2210, 5)


In [7]:
print("Gain file shape:", df_gain.shape)
print("PMF file shape :", df_pmf.shape)
print("UGF file shape :", df_ugf.shape)

display(df_gain.head())
display(df_pmf.head())
display(df_ugf.head())


Gain file shape: (2210, 5)
PMF file shape : (2210, 5)
UGF file shape : (2210, 5)


,a,b,c,d,gain
0,0.000011,0.000132,0.000002,0.000038,18.742711
1,0.000011,0.000132,0.000002,0.000043,20.022486
2,0.000011,0.000132,0.000002,0.000047,21.220200
3,0.000011,0.000132,0.000002,0.000051,22.355053
4,0.000011,0.000132,0.000002,0.000056,23.443796


,a,b,c,d,pm
0,0.000011,0.000132,0.000002,0.000038,87.258927
1,0.000011,0.000132,0.000002,0.000043,88.484256
2,0.000011,0.000132,0.000002,0.000047,89.624073
3,0.000011,0.000132,0.000002,0.000051,90.908162
4,0.000011,0.000132,0.000002,0.000056,92.119695


,a,b,c,d,ugf
0,0.000011,0.000132,0.000002,0.000038,12859450.19
1,0.000011,0.000132,0.000002,0.000043,14297892.05
2,0.000011,0.000132,0.000002,0.000047,15512936.51
3,0.000011,0.000132,0.000002,0.000051,17467633.46
4,0.000011,0.000132,0.000002,0.000056,19572012.89


In [8]:
print(df_gain.columns)
print(df_pmf.columns)
print(df_ugf.columns)


Index(['a', 'b', 'c', 'd', 'gain'], dtype='object')
Index(['a', 'b', 'c', 'd', 'pm'], dtype='object')
Index(['a', 'b', 'c', 'd', 'ugf'], dtype='object')


In [9]:
key_cols = ["a", "b", "c", "d"]

print("Gain duplicates:", df_gain.duplicated(key_cols).sum())
print("PMF duplicates :", df_pmf.duplicated(key_cols).sum())
print("UGF duplicates :", df_ugf.duplicated(key_cols).sum())


Gain duplicates: 0
PMF duplicates : 0
UGF duplicates : 0


In [10]:
df_gain[df_gain.duplicated(key_cols, keep=False)] \
    .sort_values(key_cols) \
    .head(10)


,a,b,c,d,gain


In [11]:
df_gain_agg = (
    df_gain
    .groupby(key_cols, as_index=False)
    .agg({"gain": "mean"})
)

df_pmf_agg = (
    df_pmf
    .groupby(key_cols, as_index=False)
    .agg({"pm": "mean"})
)

df_ugf_agg = (
    df_ugf
    .groupby(key_cols, as_index=False)
    .agg({"ugf": "mean"})
)


In [12]:
df_gain_agg.shape

(2210, 5)

In [13]:
df_merged = (
    df_gain_agg
    .merge(df_pmf_agg, on=key_cols)
    .merge(df_ugf_agg, on=key_cols)
)

df_merged.shape


(2210, 7)

In [14]:
df_merged.head()

,a,b,c,d,gain,pm,ugf
0,0.000005,0.000045,0.000002,0.000038,18.910978,86.235326,12172445.04
1,0.000005,0.000045,0.000002,0.000043,20.179316,87.490481,13642290.47
2,0.000005,0.000045,0.000002,0.000047,21.363141,88.652563,14882355.20
3,0.000005,0.000045,0.000002,0.000051,22.482726,89.777712,16073390.80
4,0.000005,0.000045,0.000002,0.000056,23.553926,91.027139,18206403.39


In [15]:
df_merged["ugf"] = df_merged["ugf"] / 1e6


In [16]:
df_merged.isna().sum()


a       0
b       0
c       0
d       0
gain    0
pm      0
ugf     0
dtype: int64

In [17]:
print("Final row count:", len(df_merged))
print("Unique (a,b,c,d):", df_merged[key_cols].drop_duplicates().shape[0])


Final row count: 2210
Unique (a,b,c,d): 2210


In [18]:
df_merged.head()

,a,b,c,d,gain,pm,ugf
0,0.000005,0.000045,0.000002,0.000038,18.910978,86.235326,12.172445
1,0.000005,0.000045,0.000002,0.000043,20.179316,87.490481,13.642290
2,0.000005,0.000045,0.000002,0.000047,21.363141,88.652563,14.882355
3,0.000005,0.000045,0.000002,0.000051,22.482726,89.777712,16.073391
4,0.000005,0.000045,0.000002,0.000056,23.553926,91.027139,18.206403


In [20]:
df_merged.to_csv("master_dataset.csv", index=False)
print("✅ Master dataset saved")


✅ Master dataset saved
